# PTM paper 3: machine-learning reproduction

This notebook reconstructs the numerical ML results used in Fig. 3 and Fig. S5. The released panel CSVs are the canonical figure data. The expensive Gen-2 repeated-OOF refit is provided but disabled by default; this keeps the notebook quick while retaining the complete fitting code. No model predicts experimental $P_g$: the target is MD-derived $HL_{sim}$.


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
print('Repository root:', ROOT)


## Final Gen-2 model

The final deployable model uses 25 non-MD descriptors, median imputation, an Extra-Trees regressor and a fold-local quantile transform of the target. Evaluation used 10 repeats of shuffled 10-fold cross-validation (`random_state=42`). The plotted prediction for each liquid is the mean of its ten held-out predictions; the horizontal bar is their sample standard deviation.


In [ ]:
FEATURES = ['mw','rho_exp','tm','tb','ri','pol','i_vol','alpha','nat','non_h',
            'n_N','n_P','n_O','n_S','n_F','n_Cl','n_Br','n_I','vp','kappa',
            'tc','pc','omega','triple','tg']
SEED, N_SPLITS, N_REPEATS = 42, 10, 10

def make_gen2_model(n_training_rows):
    n_quantiles = max(16, min(1000, max(2, int(n_training_rows) - 1)))
    regressor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', ExtraTreesRegressor(n_estimators=800, max_features=.6,
             min_samples_leaf=1, min_samples_split=3, bootstrap=False,
             random_state=SEED, n_jobs=-1))])
    return TransformedTargetRegressor(
        regressor=regressor,
        transformer=QuantileTransformer(output_distribution='normal',
             n_quantiles=n_quantiles, subsample=1_000_000, random_state=SEED))

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return dict(n=len(y), r2=r2_score(y,p),
                rmse_gpa=np.sqrt(mean_squared_error(y,p)),
                mae_gpa=mean_absolute_error(y,p),
                pearson_r=np.corrcoef(y,p)[0,1],
                spearman_rho=spearmanr(y,p).statistic)

dev = pd.read_csv(DATA/'ml/gen2_development_157.csv')
released_oof = pd.read_csv(DATA/'ml/gen2_repeated_oof_predictions.csv')
assert len(dev) == len(released_oof) == 157
metrics(released_oof.observed_hl_sim, released_oof.oof_prediction)


In [ ]:
# Set True to rerun all 100 Extra-Trees fits. Use the pinned environment for
# closest numerical reproducibility across scikit-learn versions.
RUN_EXPENSIVE_GEN2_OOF = False
if RUN_EXPENSIVE_GEN2_OOF:
    X = dev[FEATURES]
    y = dev.hl_sim_forcefield_mean_gpa.to_numpy(float)
    predictions = np.full((len(dev), N_REPEATS), np.nan)
    seen = np.zeros(len(dev), dtype=int)
    cv = RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    for fold, (train, test) in enumerate(cv.split(X, y), 1):
        model = make_gen2_model(len(train))  # transform is fitted inside this fold
        model.fit(X.iloc[train], y[train])
        for row, value in zip(test, model.predict(X.iloc[test])):
            predictions[row, seen[row]] = value; seen[row] += 1
        if fold % 10 == 0: print(f'{fold}/100 folds')
    recomputed = predictions.mean(axis=1)
    print(metrics(y, recomputed))
    print('Maximum difference from released artifact:',
          np.max(np.abs(recomputed - released_oof.oof_prediction.to_numpy())))


## Figure 3 data and numerical checks

Hydrogen peroxide is excluded from panel (a), as stated in the figure. Panel (b) contains 264 genuinely external PubChem candidates and marks the 17 selected for prospective MD. Panel (c) compares those pre-MD predictions to the later force-field results.


In [ ]:
fig3 = DATA/'figures/main/Fig3'
a = pd.read_csv(fig3/'panel_a_oof_predictions.csv')
b = pd.read_csv(fig3/'panel_b_external_screen_ecdf.csv')
c = pd.read_csv(fig3/'panel_c_prospective_validation.csv')
print('Panel counts:', len(a), len(b), len(c), '; selected in b:', int(b.selected_for_set3.sum()))
print('Fig. 3a:', metrics(a.observed_hl_sim, a.oof_prediction))
print('Fig. 3c:', metrics(c.observed_sigma1_mean, c.sigma_def1_pred_et_qttr))
assert (len(a),len(b),len(c),int(b.selected_for_set3.sum())) == (156,264,17,17)


In [ ]:
# Compact reconstruction of Fig. 3 from the exact released panel data.
purple, blue, orange, grey = '#72519A', '#8D9BC2', '#F0A13E', '#888888'
fig, ax = plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
ax[0].errorbar(a.oof_prediction,a.observed_hl_sim,xerr=a.oof_prediction_sd,
               fmt='o',ms=6,mfc=purple,mec='#333',ecolor='#B9AED0',alpha=.9,lw=.8)
ax[0].plot([0,7],[0,7],'--',color=grey); ax[0].set(xlim=(0,7),ylim=(0,7),
               xlabel=r'$HL_{sim,pred}$ (GPa)',ylabel=r'$HL_{sim}$ (GPa)')
ax[1].step(b.sigma_def1_pred_et_qttr,b.empirical_cumulative_fraction,where='post',color=purple)
s=b.selected_for_set3.astype(bool)
ax[1].scatter(b.loc[s,'sigma_def1_pred_et_qttr'],b.loc[s,'empirical_cumulative_fraction'],
              s=55,color=purple,edgecolor='#333',zorder=3)
ax[1].set(xlabel=r'$HL_{sim,pred}$ (GPa)',ylabel=r'$\widehat{F}$',ylim=(0,1.03))
ax[2].plot([0,10],[0,10],'--',color=grey)
for _,r in c.iterrows():
    ends=[v for v in [r.sigma_1_cgen,r.sigma_1_opls] if pd.notna(v)]
    if len(ends)==2: ax[2].plot([r.sigma_def1_pred_et_qttr]*2,ends,color='#B9AED0',lw=1)
ax[2].scatter(c.sigma_def1_pred_et_qttr,c.observed_sigma1_mean,s=65,color=purple,edgecolor='#333',zorder=3)
ax[2].scatter(c.sigma_def1_pred_et_qttr,c.sigma_1_cgen,s=25,color=blue,edgecolor='#333',zorder=4)
ax[2].scatter(c.sigma_def1_pred_et_qttr,c.sigma_1_opls,s=32,marker='D',color=orange,edgecolor='#333',zorder=4)
ax[2].set(xlim=(0,10),ylim=(0,10),xlabel=r'$HL_{sim,pred}$ (GPa)',ylabel=r'$HL_{sim}$ (GPa)')
for i,x in enumerate(ax): x.text(.02,.96,f'({chr(97+i)})',transform=x.transAxes,va='top',fontweight='bold'); x.spines[['top','right']].set_visible(False)
plt.show()


## Gen-1 versus Gen-2 (Fig. S5)

Gen-1 and Gen-2 differ in population, descriptors, model class and target definition, so Fig. S5 is a lineage comparison rather than a controlled ablation. Gen-1 OOF values are the documented leakage-controlled reconstruction; exact prospective predictions were recovered from saved historical outputs.


In [ ]:
s5 = DATA/'figures/supplementary/FigS5'
g1o=pd.read_csv(s5/'panel_a_gen1_oof.csv'); g2o=pd.read_csv(s5/'panel_b_gen2_oof.csv')
g1p=pd.read_csv(s5/'panel_d_gen1_set2.csv'); g2p=pd.read_csv(s5/'panel_e_gen2_set3.csv')
summary = {
 'Gen-1 OOF':metrics(g1o.observed_hl_sim,g1o.oof_prediction),
 'Gen-2 OOF':metrics(g2o.observed_hl_sim,g2o.oof_prediction),
 'Gen-1 prospective Set 2':metrics(g1p.observed_hl_sim,g1p.predicted_hl_sim),
 'Gen-2 prospective Set 3':metrics(g2p.observed_hl_sim,g2p.predicted_hl_sim)}
pd.DataFrame(summary).T


## Reproducibility boundary

The panel CSVs, identifiers, split definitions, repeated OOF predictions and prospective predictions are released verbatim. Retraining tree ensembles or neural networks can vary at the last digits with library/hardware changes; therefore the saved OOF artifacts are the authoritative numerical inputs to the published figures. `validation/VALIDATION.md` documents the independent checks against the supplied final PDFs and manuscript annotations.
